# Graph Programming — 06: Advanced Topics

Three powerful techniques that appear frequently in harder graph problems:

1. **Union-Find (Disjoint Set Union)** — efficiently track connected components
2. **Dijkstra's Algorithm** — shortest path in weighted graphs
3. **Bipartite Graph Check** — 2-color problem

---

## Part 1: Union-Find (DSU)

Union-Find answers: **"Are these two nodes in the same component?"** in near O(1) time.

```
Operations:
  find(x)    → root/representative of x's component
  union(x,y) → merge the two components
```

Two optimizations make it nearly O(1) amortized:
- **Path compression**: During find, point directly to root
- **Union by rank**: Attach smaller tree under larger tree

In [ ]:
# ============================================================
# UNION-FIND TEMPLATE
# ============================================================

class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))   # each node is its own root
        self.rank = [0] * n
        self.components = n            # track number of components

    def find(self, x):
        # Path compression: point directly to root
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False   # already connected
        # Union by rank: attach smaller under larger
        if self.rank[rx] < self.rank[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        if self.rank[rx] == self.rank[ry]:
            self.rank[rx] += 1
        self.components -= 1
        return True   # merged two different components

    def connected(self, x, y):
        return self.find(x) == self.find(y)

# Demo
uf = UnionFind(6)
uf.union(0, 1)
uf.union(1, 2)
uf.union(3, 4)
print("Connected 0-2?", uf.connected(0, 2))   # True
print("Connected 0-3?", uf.connected(0, 3))   # False
print("Components:", uf.components)            # 3 ({0,1,2}, {3,4}, {5})

In [ ]:
# === LC 684: Redundant Connection ===
# Find the edge that creates a cycle in an undirected graph.
# (n nodes, n edges — exactly one cycle)

def findRedundantConnection(edges):
    n = len(edges)
    uf = UnionFind(n + 1)   # nodes are 1-indexed

    for u, v in edges:
        if not uf.union(u, v):   # union returns False if already connected
            return [u, v]        # this edge creates a cycle!

    return []

print(findRedundantConnection([[1,2],[1,3],[2,3]]))   # [2,3]
print(findRedundantConnection([[1,2],[2,3],[3,4],[1,4],[1,5]]))  # [1,4]

In [ ]:
# === Number of Islands via Union-Find ===
# Alternative to DFS/BFS approach

def numIslands_UF(grid):
    rows, cols = len(grid), len(grid[0])
    uf = UnionFind(rows * cols)
    count = 0

    def idx(r, c):
        return r * cols + c   # 2D → 1D index

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == '1':
                count += 1
                for dr, dc in [(0,1),(1,0)]:   # only check right and down (avoid duplicates)
                    nr, nc = r+dr, c+dc
                    if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == '1':
                        if uf.union(idx(r,c), idx(nr,nc)):
                            count -= 1   # merged two islands

    return count

grid = [['1','1','0'],['0','1','0'],['0','0','1']]
print(numIslands_UF(grid))  # 2

---
## Part 2: Dijkstra's Algorithm — Shortest Path (Weighted)

BFS works for shortest path in **unweighted** graphs.  
For **weighted** graphs, use Dijkstra's.

```
Core idea:
- Min-heap (priority queue) ordered by distance
- Always process the closest unvisited node
- Update neighbors if we found a shorter path

Time: O((V + E) log V)
```

**Does NOT work with negative weights.** Use Bellman-Ford for that.

In [ ]:
import heapq
from collections import defaultdict

# ============================================================
# DIJKSTRA'S TEMPLATE
# ============================================================

def dijkstra(graph, start, end=None):
    """
    graph: dict of {node: [(neighbor, weight), ...]}
    Returns: dist dict (shortest distance from start to all nodes)
    """
    dist = {node: float('inf') for node in graph}
    dist[start] = 0
    heap = [(0, start)]   # (distance, node)

    while heap:
        d, node = heapq.heappop(heap)

        if d > dist[node]:   # stale entry in heap — skip
            continue
        if node == end:      # early exit if only care about one target
            break

        for neighbor, weight in graph[node]:
            new_dist = dist[node] + weight
            if new_dist < dist[neighbor]:
                dist[neighbor] = new_dist
                heapq.heappush(heap, (new_dist, neighbor))

    return dist

# Weighted graph:
#   0 --4-- 1
#   |       |
#   1       2
#   |       |
#   2 --1-- 3

graph = {
    0: [(1, 4), (2, 1)],
    1: [(0, 4), (3, 2)],
    2: [(0, 1), (3, 5)],
    3: [(1, 2), (2, 5)]
}

distances = dijkstra(graph, 0)
print("Shortest distances from 0:", distances)
# 0→0:0, 0→1:4, 0→2:1, 0→3:6 (via 0→2→1... wait: 0→1=4, 0→2=1, 0→2→3=1+5=6 or 0→1→3=4+2=6)

In [ ]:
# === LC 743: Network Delay Time ===
# Given n nodes, edges with times, find minimum time for ALL nodes to receive signal
# from source k. If not all reachable, return -1.

def networkDelayTime(times, n, k):
    graph = defaultdict(list)
    for u, v, w in times:
        graph[u].append((v, w))

    dist = {i: float('inf') for i in range(1, n+1)}
    dist[k] = 0
    heap = [(0, k)]

    while heap:
        d, node = heapq.heappop(heap)
        if d > dist[node]:
            continue
        for neighbor, weight in graph[node]:
            new_d = d + weight
            if new_d < dist[neighbor]:
                dist[neighbor] = new_d
                heapq.heappush(heap, (new_d, neighbor))

    max_time = max(dist.values())
    return max_time if max_time != float('inf') else -1

print(networkDelayTime([[2,1,1],[2,3,1],[3,4,1]], 4, 2))  # 2
print(networkDelayTime([[1,2,1]], 2, 2))                   # -1 (2 can't reach 1)

In [ ]:
# === LC 787: Cheapest Flights Within K Stops ===
# Dijkstra variant where state = (cost, node, stops_remaining)

def findCheapestPrice(n, flights, src, dst, k):
    graph = defaultdict(list)
    for u, v, w in flights:
        graph[u].append((v, w))

    # (cost, node, stops_left)
    heap = [(0, src, k+1)]
    visited = {}   # node → best stops_left we've seen it with

    while heap:
        cost, node, stops = heapq.heappop(heap)
        if node == dst:
            return cost
        if stops == 0:
            continue
        if node in visited and visited[node] >= stops:
            continue
        visited[node] = stops
        for neighbor, price in graph[node]:
            heapq.heappush(heap, (cost + price, neighbor, stops - 1))

    return -1

print(findCheapestPrice(4, [[0,1,100],[1,2,100],[2,0,100],[1,3,600],[2,3,200]], 0, 3, 1))  # 700

---
## Part 3: Bipartite Graph Check

A graph is **bipartite** if you can color all nodes with 2 colors such that no two adjacent nodes share the same color.

Equivalently: a graph is bipartite if and only if it contains **no odd-length cycles**.

```
Bipartite:          Not bipartite:
  A - B               A
  |   |              / \
  C - D             B - C
   (square)        (triangle — odd cycle)
```

**Algorithm:** BFS/DFS trying to 2-color. If you find two neighbors with the same color → not bipartite.

In [ ]:
# === LC 785: Is Graph Bipartite? ===

def isBipartite(graph):
    n = len(graph)
    color = [-1] * n   # -1 = uncolored, 0 = red, 1 = blue

    def bfs(start):
        from collections import deque
        queue = deque([start])
        color[start] = 0
        while queue:
            node = queue.popleft()
            for neighbor in graph[node]:
                if color[neighbor] == -1:              # uncolored: give opposite color
                    color[neighbor] = 1 - color[node]
                    queue.append(neighbor)
                elif color[neighbor] == color[node]:   # same color as parent → not bipartite!
                    return False
        return True

    for node in range(n):
        if color[node] == -1:
            if not bfs(node):
                return False
    return True

# Square graph: 0-1-2-3-0
print(isBipartite([[1,3],[0,2],[1,3],[0,2]]))   # True
# Triangle: 0-1-2-0
print(isBipartite([[1,2],[0,2],[0,1]]))         # False

---
## Summary: Which Algorithm to Use?

```
PROBLEM TYPE                          → ALGORITHM
─────────────────────────────────────────────────────────
Shortest path, unweighted graph       → BFS
Shortest path, weighted graph         → Dijkstra (min-heap)
Shortest path, negative weights       → Bellman-Ford
All pairs shortest paths              → Floyd-Warshall
Detect cycle, undirected              → DFS (track parent)
Detect cycle, directed                → DFS (3-color: 0/1/2)
Topological order                     → Kahn's BFS (in-degree)
Count / find connected components     → DFS or BFS from each node
Merge components, track membership    → Union-Find
Check if 2-colorable                  → BFS bipartite check
Grid: flood fill, islands, regions    → DFS or BFS on grid
Multi-source spread (rotting, walls)  → Multi-source BFS
```

## Graph Cheatsheet — Complexity

| Algorithm | Time | Space |
|---|---|---|
| DFS / BFS | O(V + E) | O(V) |
| Dijkstra | O((V+E) log V) | O(V) |
| Topological Sort | O(V + E) | O(V) |
| Union-Find | O(α(n)) ≈ O(1) | O(V) |

Where V = vertices, E = edges, α = inverse Ackermann (effectively constant).